### Import Dependencies

In [1]:
import openai
from qdrant_client import QdrantClient

### Embedding function

In [2]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

### Retrieval function

In [3]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [4]:
def retrieve_data(query, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

In [5]:
retrieved_context = retrieve_data("Do you have a USB connectable fan for hot summers.", k=10)

In [6]:
retrieved_context

{'retrieved_context_ids': ['B09QWDNQH9',
  'B0C61QHRB6',
  'B0BHL5B62F',
  'B07VDPWPLQ',
  'B0BRYFBCRF',
  'B09XBGM4GB',
  'B0B53NTZJN',
  'B0B72B3NRC',
  'B0BG2ZDW9P',
  'B0BSR51DLT'],
 'retrieved_context': ['Radiate Like This[LP] ',
  'If ',
  'Sheer Heart Attack[Half-Speed LP] ',
  'The Complete Harry Potter Film Music Collection ',
  'Bluey Dance Mode Orange ',
  'Licked Live In NYC[2 CD] ',
  'Sonder ',
  'The Mars Volta ',
  'Love ',
  'RökFlöte '],
 'similarity_scores': [0.14792387,
  0.14762427,
  0.13324565,
  0.13263427,
  0.1299313,
  0.12518246,
  0.12182141,
  0.1195098,
  0.119071916,
  0.11800332],
 'retrieved_context_ratings': [4.7,
  4.6,
  4.8,
  4.8,
  4.8,
  4.7,
  4.8,
  4.6,
  4.8,
  4.3]}

### Format retrieved context function

In [8]:
def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context

In [9]:
preprocessed_context = process_context(retrieved_context)

In [10]:
print(preprocessed_context)

- ID: B09QWDNQH9, rating: 4.7, description: Radiate Like This[LP] 
- ID: B0C61QHRB6, rating: 4.6, description: If 
- ID: B0BHL5B62F, rating: 4.8, description: Sheer Heart Attack[Half-Speed LP] 
- ID: B07VDPWPLQ, rating: 4.8, description: The Complete Harry Potter Film Music Collection 
- ID: B0BRYFBCRF, rating: 4.8, description: Bluey Dance Mode Orange 
- ID: B09XBGM4GB, rating: 4.7, description: Licked Live In NYC[2 CD] 
- ID: B0B53NTZJN, rating: 4.8, description: Sonder 
- ID: B0B72B3NRC, rating: 4.6, description: The Mars Volta 
- ID: B0BG2ZDW9P, rating: 4.8, description: Love 
- ID: B0BSR51DLT, rating: 4.3, description: RökFlöte 



### Create prompt template function

In [11]:
def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{question}    
"""

    return prompt

In [12]:
prompt = build_prompt(preprocessed_context, "Do you have a USB connectable fan for hot summers?")

In [13]:
print(prompt)


You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
- ID: B09QWDNQH9, rating: 4.7, description: Radiate Like This[LP] 
- ID: B0C61QHRB6, rating: 4.6, description: If 
- ID: B0BHL5B62F, rating: 4.8, description: Sheer Heart Attack[Half-Speed LP] 
- ID: B07VDPWPLQ, rating: 4.8, description: The Complete Harry Potter Film Music Collection 
- ID: B0BRYFBCRF, rating: 4.8, description: Bluey Dance Mode Orange 
- ID: B09XBGM4GB, rating: 4.7, description: Licked Live In NYC[2 CD] 
- ID: B0B53NTZJN, rating: 4.8, description: Sonder 
- ID: B0B72B3NRC, rating: 4.6, description: The Mars Volta 
- ID: B0BG2ZDW9P, rating: 4.8, description: Love 
- ID: B0BSR51DLT, rating: 4.3, description: RökFlöte 


Question:
Do you have a USB conne

### Generate answer function

In [17]:
def generate_answer(prompt):

    response = openai.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[
            {"role": "system", "content": prompt}
        ],
        reasoning_effort="none"
    )

    return response.choices[0].message.content

In [16]:
print(generate_answer(prompt))

I don’t have any USB connectable fans in the available products list.

If you want, tell me what size/style you’re looking for (desktop, clip-on, rechargeable vs. USB-only), and I can help check what options might be available.


### Combined RAG pipeline

In [18]:
def rag_pipeline(question, top_k=5):

    retrieved_context = retrieve_data(question, k=top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    return answer
    

In [19]:
print(rag_pipeline("Do you have a USB connectable fan for hot summers?"))

I don’t have any USB-connectable fan listed among the available products. What I do have are items like vinyl records (e.g., Sheer Heart Attack, The Complete Harry Potter Film Music Collection) and a Bluey toy (“Bluey Dance Mode Orange”).


In [20]:
print(rag_pipeline("Could you suggest me some earphones? I am only interested in the ones that have above 4 rating.", 10))

Sure—here are the available options with a rating above 4:

1) Bluey Dance Mode Orange (4.8) — ID: B09ZTFPVNB  
2) The Complete Harry Potter Film Music Collection (4.8) — ID: B07VDPWPLQ  
3) All My Friends: Celebrating The Songs & Voice Of Gregg Allman [4 LP] (4.8) — ID: B0B61K8PQM  
4) Beatopia[LP] (4.8) — ID: B09TMYXDMG  
5) 5-STAR VER. A (4.8) — ID: B0C44ZF8Y7  
6) Radiate Like This[LP] (4.7) — ID: B09QWDNQH9  
7) Songs About You (4.8) — ID: B09Y4X2XTS  
8) Against The Odds: 1974-1982[8 CD] (4.8) — ID: B0B1TPKCPB  

If you tell me whether you want wired or wireless earphones (and your budget), I can narrow it further.


In [21]:
print(rag_pipeline("Could you suggest me some earphones? I am only interested in the ones that have bellow 4 rating.", 10))

I can’t suggest earphones from the available items because none of the available products are earphones, and all listed items have ratings of 4.6 or higher (so there aren’t any with a rating below 4).
